In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load Dataset
from datasets import load_dataset

train_dataset = load_dataset("trl-lib/kto-mix-14k", split="train")

train_dataset

Dataset({
    features: ['prompt', 'completion', 'label'],
    num_rows: 13500
})

In [3]:
# view sample dataset
import pprint

pprint.pprint(train_dataset[0])

{'completion': [{'content': ' Yes, the information you found on Google is '
                            'correct. Julio César Chávez holds several records '
                            'related to world title defenses and victories, '
                            'and he is considered one of the greatest boxers '
                            'in history. Here is a detailed answer to your '
                            'question:\n'
                            '\n'
                            'Julio César Chávez was born on July 12, 1962, in '
                            'Ciudad Obregón, Sonora, Mexico. He began boxing '
                            'at a young age and quickly made a name for '
                            'himself in the sport, winning his first world '
                            'title in 1984 when he defeated Mario Miranda for '
                            'the WBC super featherweight championship.\n'
                            '\n'
                            'Over the

In [4]:
# Load model and tokenizer

from trl import KTOConfig, KTOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1574.84it/s]


In [ ]:
# examine the model architecture
pprint.pprint(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
# examine the tokenizer
pprint.pprint(tokenizer)

Qwen2Tokenizer(name_or_path='Qwen/Qwen2-0.5B-Instruct', vocab_size=151643, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [ ]:
# Training configuration and trainer setup

training_args = KTOConfig(
    output_dir="Qwen2-0.5B-KTO",
    num_train_epochs=1,
    per_device_train_batch_size=2,
)

trainer = KTOTrainer(
    model=model, 
    args=training_args, 
    processing_class=tokenizer, 
    train_dataset=train_dataset)

# Start training
trainer.train()

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Dropping fully truncated examples from train dataset: 100%|██████████| 13500/13500 [00:00<00:00, 17431.75 examples/s]
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 321.19it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.508800
20,0.500367
30,0.504479
40,0.498924
50,0.501146
60,0.499376
70,0.504729
80,0.505426
90,0.504544
100,0.503076


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]
